In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.svm import SVC

In [2]:
import sys
sys.path.append("/Users/mariahloehr/IICD/IICD/feature_importance")

In [3]:
import locomp
from locomp import *
from locomp.MLmodels import *
from locomp.util_locomp import *
import itertools
import importlib
from sklearn.base import BaseEstimator, RegressorMixin, clone
import itertools
from functools import partial
import multiprocessing as mp
import re

import functions_case as il
import importlib

In [4]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Separate features and target
X = df.drop(columns=['phase'])
y = le.fit_transform(df['phase'])
perturbation_col = X.columns.get_loc("Metadata_well")

feature_names = X.columns.tolist()
feature_pairs = list(itertools.combinations(range(X.shape[1]), 2))
X = X.to_numpy()
#y = y.to_numpy()

# Split data into train and test sets (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=949, stratify=y)

In [5]:
DecisionTreeClass = DecisionTreeClassifier(max_depth = 50, 
                                 max_features=10,
                                 random_state=949
                                 )

In [6]:
xgbclass = GradientBoostingClassifier(
        n_estimators=10,       # fixed boosting rounds
        learning_rate=0.1, # hyperparameters from XGB model
        max_depth=7,
        random_state=949
    )

In [7]:
# Define RBF-kernel SVM
svmclass = SVC(kernel='rbf', C = 400, gamma = 0.01, probability=True, random_state=949
              )

In [8]:
J1 = 0
J2 = 1
m_ratio = 0.2
n_ratio = 0.2
B = 10
models = (DecisionTreeClass, xgbclass, svmclass)
perturbation_col = perturbation_col

In [9]:
def get_mp_metric(X, Y, n_ratio, m_ratio, B,
                   models, perturbation_col):
    """
    Computes interaction metrics (iLOCO) for multiple feature pairs using an ensemble model.

    Parameters:
    X (numpy.ndarray): Feature matrix.
    Y (numpy.ndarray): Target variable.
    n_ratio (float): Ratio parameter for ensemble fitting.
    m_ratio (float): Ratio parameter for ensemble fitting.
    B (int): Number of bootstrap samples.
    model (object): Trained model for interaction computation.
    feature_pairs (list of tuples): List of tuples where each tuple contains two feature indices (j1, j2) for the feature pair.

    Returns:
    dict: Dictionary where keys are feature pairs (j1, j2) and values are dictionaries containing:
          - 'iloco': The mean interaction value.
          - 'iloco_max': The maximum interaction value, ensuring non-negative.
          - 'iloco_ratio': The ratio of iloco to the mean of r.
    """
    # Create and fit an ensemble model
    ensemble = il.Ensemble(models).fit(X, Y, n_ratio, m_ratio, B, perturbation_col = perturbation_col)

    # Compute feature interactions for the given list of feature pairs
    result = il.mp_and_featureInteractions_class(X, Y, n_ratio, m_ratio, B, models, 
                                                 feature_pairs, perturbation_col, bonferroni=True)

    return ensemble

In [10]:
mp_results = []
mp_ci = []
mp_results = get_mp_metric(X, y, n_ratio, m_ratio, B, models, perturbation_col=perturbation_col)
mp_scores = [mp_results[(j, k)]['iloco'] for (j, k) in feature_pairs]
mp_ci = [mp_results[(j, k)]["ci"] for (j, k) in feature_pairs]

AssertionError: 

In [14]:
len(mp_results.ensemble)        # number of minipatches/models

10

In [15]:
mp_results.ensemble[:10]         # first three models

[DecisionTreeClassifier(max_depth=50, max_features=10, random_state=949),
 GradientBoostingClassifier(max_depth=7, n_estimators=10, random_state=949),
 GradientBoostingClassifier(max_depth=7, n_estimators=10, random_state=949),
 SVC(C=400, gamma=0.01, probability=True, random_state=949),
 SVC(C=400, gamma=0.01, probability=True, random_state=949),
 DecisionTreeClassifier(max_depth=50, max_features=10, random_state=949),
 GradientBoostingClassifier(max_depth=7, n_estimators=10, random_state=949),
 DecisionTreeClassifier(max_depth=50, max_features=10, random_state=949),
 GradientBoostingClassifier(max_depth=7, n_estimators=10, random_state=949),
 GradientBoostingClassifier(max_depth=7, n_estimators=10, random_state=949)]

In [16]:
# which features were in patch b
np.where(mp_results.mp_features[:, 5])[0]

array([ 2,  6, 17, 20])

In [17]:
preds = mp_results.predict(X)
preds

array([[1., 0., 1., ..., 3., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(64502, 10))